In [ ]:
# Install Required Libraries 

!pip install transformers accelerate bitsandbytes gradio pandas 

In [ ]:
# Import Libraries 
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import gradio as gr


In [ ]:
# Load an Open Source Model 
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True   # quantization
)

In [ ]:
# Create Text Generation Pipeline 

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    temperature=0.7
)

In [ ]:
# Synthetic Data Generator Function 
def generate_synthetic_data(domain, num_records):

    prompt = f"""
    Generate {num_records} synthetic records for {domain}.
    Return data in CSV format.

    Example columns:
    id,name,department,location,salary
    """

    result = generator(prompt)[0]["generated_text"]

    return result

In [ ]:
# Convert Generated Text to DataFrame 

def generate_dataframe(domain, num_records):

    text = generate_synthetic_data(domain, num_records)

    try:
        df = pd.read_csv(pd.compat.StringIO(text))
        return df
    except:
        return text

In [ ]:
# Create Gradio UI 
def ui_generate(domain, records):
    data = generate_synthetic_data(domain, records)
    return data

interface = gr.Interface(
    fn=ui_generate,
    inputs=[
        gr.Textbox(label="Dataset Description"),
        gr.Slider(1,50,step=1,label="Number of Records")
    ],
    outputs="text",
    title="Synthetic Dataset Generator",
    description="Generate synthetic datasets using open-source LLMs"
)

interface.launch()
